What are types of scopes in pytest .

@pytest.fixture(scope="function")   # created/destroyed per test
@pytest.fixture(scope="class")      # created/destroyed per test class
@pytest.fixture(scope="module")     # created/destroyed per test file
@pytest.fixture(scope="session")    # created/destroyed once per full run


FUNCTION SCOPE
==============
Test 1 → [CREATE fixture] → test runs → [DESTROY fixture]
Test 2 → [CREATE fixture] → test runs → [DESTROY fixture]
Test 3 → [CREATE fixture] → test runs → [DESTROY fixture]

CLASS SCOPE
===========
Class A:
    Test 1 → [CREATE fixture] → test runs
    Test 2 →                  → test runs
    Test 3 →                  → test runs → [DESTROY fixture]
Class B:
    Test 1 → [CREATE fixture] → test runs
    Test 2 →                  → test runs → [DESTROY fixture]

MODULE SCOPE
============
File test_reset.py:
    Class A Test 1 → [CREATE fixture] → test runs
    Class A Test 2 →                  → test runs
    Class B Test 1 →                  → test runs
    Function Test  →                  → test runs → [DESTROY fixture]
File test_identify.py:
    Test 1         → [CREATE fixture] → test runs → [DESTROY fixture]

SESSION SCOPE
=============
pytest run begins → [CREATE fixture]
    test_identify.py  → all tests reuse same fixture
    test_reset.py     → all tests reuse same fixture
    test_firmware.py  → all tests reuse same fixture
pytest run ends   → [DESTROY fixture]


@pytest.fixture(scope="session")
def shelf_connection():
    print("Connecting to shelf...")   # happens exactly once
    conn = ShelfConnection()
    conn.connect()
    yield conn
    conn.disconnect()
    print("Disconnected from shelf")  # happens exactly once

## Session is pytest session, pytest can execute many python scripts in form of python files. Once the instance of pytest session is created a fixture is created and once it goes out of session the pytest session goes out of scope.

    pytest run starts
    → connect to shelf ONCE

    test_identify.py
        test_identify_controller     → reuses connection → test
        test_identify_namespace      → reuses connection → test
        test_identify_firmware_rev   → reuses connection → test
    test_reset.py
        test_nssr_recovery           → reuses connection → test
        test_flr_recovery            → reuses connection → test
        test_perst_recovery          → reuses connection → test
        test_power_cycle             → reuses connection → test
    test_firmware_update.py
        test_bdfu_basic              → reuses connection → test
        test_bdfu_with_io            → reuses connection → test
        test_activation_rollback     → reuses connection → test

    → disconnect from shelf ONCE

Total shelf connections: 1
Total time saved: 270 seconds


# Where should the session scoped fixture should be defined ?
# It should be defined at the root of all the tests under "conftest.py" file so that pytest can execute it before launching any other python script.

tests/
    conftest.py              ← session-scoped shelf_connection lives HERE
    test_identify.py
    test_reset.py
    test_firmware_update.py


    The contents of conftest.py should contain below :

    # conftest.py
import pytest

@pytest.fixture(scope="session")
def shelf_connection():
    conn = ShelfConnection()
    conn.connect()
    yield conn
    conn.disconnect()
    
================
The Real Rule:
================
State pollution can happen whenever a fixture is shared across more than one test and one test modifies the fixture's internal state before the next test uses it.

Scope	Shared Across	State Pollution Risk
function	Not shared — new instance per test	No risk — each test gets a fresh object
class	All tests within one class	Yes — if Test 2 modifies state, Test 3 sees it
module	All tests within one file	Yes — any test can dirty the state for subsequent tests
session	All tests in the entire run	Yes — highest blast radius because it spans all files

In [8]:
# Write a decorator @timer that measures and prints how long a function takes to execute
import time
def timer(fnc):
    def print_time_taken(*args, **kwargs):
        start_time=time.perf_counter()
        temp=fnc(*args, **kwargs)
        end_time=time.perf_counter()
        print(end_time-start_time)
        return temp

    return print_time_taken 

@timer
def slow_function():
    time.sleep(2)
    return "done"

low=slow_function()
print(low)  

2.0002223000046797
done


In [12]:
# Problem 2 — Decorator with Arguments
#Concept: Decorator factory — a decorator that takes its own parameters

#Write a decorator @retry(times=3) that retries a function up to N times 
# if it raises an exception, and raises the final exception if all retries fail.

import time
from functools import wraps
def retry(times=3):
    def actual_decorator(func):

        @wraps(func)
        def func_wrapper(*args,**kwargs):
            for i in range(times):
                try:
                    return func(*args,**kwargs)
                except Exception as e:
                    if i == times - 1:
                        raise e
        return func_wrapper
    return actual_decorator


@retry(times=3)
def unstable_api_call():
    if time.time() % 2 < 1:  # Simulate a failure half the time
        raise ValueError("API call failed")
    return "API call succeeded"


print(unstable_api_call())











API call succeeded


In [13]:
# Problem 3 — Logging Decorator
#Concept: Inspecting function arguments inside a decorator

#Write a decorator @log_call that prints the function name, 
# the arguments it was called with, and the return value every time it executes.
from functools import wraps    
def log_call(func):
    @wraps(func)
    def func_wrapper(*args,**kwargs):
        print(f"Function name :{func.__name__} ")
        print(f"Arguments: args={args}, kwargs={kwargs}")
        result = func(*args, **kwargs)
        print(f"Return value: {result}")
        return result
    return func_wrapper

@log_call
def add(a, b):
    return a + b

print(add(3, 5))






Function name :add 
Arguments: args=(3, 5), kwargs={}
Return value: 8
8


In [14]:
#Problem 4 — Access Control / Authorization
#Concept: Conditional execution based on state
#Write a decorator @require_auth that checks whether 
# a global variable current_user has a role of "admin" 
# before allowing the function to execute. If not, print 
# "Access denied" and return None.
from functools import wraps

current_user = {"name": "Mukesh", "role": "admin"}

def require_auth(func):
    @wraps(func)
    def func_wrapper(*args,**kwargs):
         
         global current_user
         if current_user.get("role") == "admin":
             return func(*args,**kwargs)
         else:
                print("Access denied")
                return None 
    return func_wrapper


@require_auth
def delete_namespace():
    print("Namespace deleted")


delete_namespace()  # Should print "Namespace deleted" since current_user is admin



Namespace deleted


In [18]:
# About frozenset in python

a = frozenset({1, 2, 3})
print(a)
b=frozenset([1, 2, 3])
print(b)




frozenset({1, 2, 3})
frozenset({1, 2, 3})


In [29]:
#Problem 5 — Memoization / Caching
#Concept: Storing state between calls using a closure

#Write a decorator @memoize that caches the return value of a function based on its arguments. If the same arguments are passed again, return the cached result without re-executing the function.

from functools import wraps
import time


def memoize(func):
    cache = {}
    @wraps(func)
    def func_wrapper(*args, **kwargs):
        key=(args,tuple(sorted(kwargs.items())))
        if key in cache:
            print(f"Cache hit , for {func.__name__} with args={args} and kwargs={kwargs}")
            return cache[key]
        result = func(*args, **kwargs)
        cache[key] = result
        return result
    func_wrapper.cache = cache  # Expose the cache for inspection
    func_wrapper.clear_cache=cache.clear
    return func_wrapper 


@memoize
def expensive_compute(n):
    
    time.sleep(1)
    
    return n * n

print(expensive_compute(4))  # This will take 1 second
print(expensive_compute(4))  # This will return immediately from cache
print(expensive_compute(5))  # This will take 1 second
print(expensive_compute.cache)
expensive_compute.clear_cache()  # Clear the cache
print(expensive_compute.cache)  # This should now be empty





16
Cache hit , for expensive_compute with args=(4,) and kwargs={}
16
25
{((4,), ()): 16, ((5,), ()): 25}
{}


In [30]:
# How to attach attributes and methods to any function in python.
# In Python functions are objects, so we can attach attributes and methods to them using the dot operator.
# For example, we can attach a method to a function as follows:
def my_function():
    print("Hello, world!")

# Attach an attribute
my_function.my_attribute = 42

# Attach a method
def my_method():
    print("This is a method attached to the function.")

my_function.my_method = my_method

# Usage
print(my_function.my_attribute)  # Output: 42
my_function.my_method()  # Output: This is a method attached to the function.


42
This is a method attached to the function.


In [26]:
# In python tupules are considered as ordered set so the order in which the tuples apper matters

# Even though they contain the exact same items, Python tuples are ordered sequences. Because the items appear in a different order, 
# Python treats them as completely separate, independent keys.

my_dict={}
key1=(('n',4),('mode','fast'))
key2=(('mode','fast'),('n',4))
my_dict[key1]="Result A"
my_dict[key2]="Result B"
print(my_dict)

# Here we can see inven though the tuples contain the same items 
# But due to change of order they are treated as different keys and hence the dictionary contains two different entries for them

# In order to avoid this we can use frozenset instead of 
# tuples as frozensets are unordered collections and hence the order of items does not matter and hence they will be treated as same keys in dictionary


my_dict={}
key1=frozenset((('n',4),('mode','fast')))
key2=frozenset((('mode','fast'),('n',4)))
my_dict[key1]="Result A"
my_dict[key2]="Result B"
print(my_dict)





{(('n', 4), ('mode', 'fast')): 'Result A', (('mode', 'fast'), ('n', 4)): 'Result B'}
{frozenset({('mode', 'fast'), ('n', 4)}): 'Result B'}


In [33]:
#Problem 6 — Stacking Multiple Decorators

#Concept: Decorator execution order
#What it tests: Understanding that decorators apply bottom-up but execute top-down. This is a classic interview question on decorator stacking order.

#Write two decorators — @bold and @italic — that wrap a string return value in HTML tags. Then apply both to a function and predict the output order before running it.

#@bold
#@italic
#def greet():
#    return "Hello"
#print(greet())

from functools import wraps

def italic(func):
    @wraps(func)
    def wrapper_function(*args,**kwargs):
        print(f"Executing {func.__name__} inside italic decorator")
        result=f"<i>{func(*args, **kwargs)}</i>"
        print(f"Finished executing {func.__name__} inside italic decorator  ")
        return result
    return wrapper_function

def bold(func):
    @wraps(func)
    def wrapper_function(*args,**kwargs):
        print(f"Executing {func.__name__} inside bold decorator")
        result = f"<b>{func(*args, **kwargs)}</b>"
        print(f"Finished executing {func.__name__} inside bold decorator")
        return result
    return wrapper_function


@bold
@italic 
def greet():
    return "Hello"

print(greet())










Executing greet inside bold decorator
Executing greet inside italic decorator
Finished executing greet inside italic decorator  
Finished executing greet inside bold decorator
<b><i>Hello</i></b>


In [2]:
import sys
!{sys.executable} -m pip install ipytest
!{sys.executable} -m pip install coverage
!{sys.executable} -m pip install pytest-cov


In [28]:
# use of pytest  fixture without autouse=True and scope=class
 
import pytest
import ipytest
import sys



ipytest.autoconfig()




@pytest.fixture(scope="class")
def setup_cleanup():
    print("Setting up resources before the test")
    yield
    print("Cleaning up resources after the test")

class TestExample:
    def test_one(self, setup_cleanup):
        print("Executing test_one")
        assert True

    def test_two(self, setup_cleanup):
        print("Executing test_two")
        assert True




ipytest.run("-v","-s" )

        
            

======================================= test session starts =======================================
platform win32 -- Python 3.13.9, pytest-8.4.2, pluggy-1.5.0
rootdir: d:\Learning\Projects\TimePass\DSA
plugins: anyio-4.10.0, cov-7.1.0
collected 4 items

t_f58b8dd5a3c941419e527b68f1402cb9.py Executing test_one
.Executing test_two
.Setting up resources before the test
Executing test_one
.Executing test_two
.Cleaning up resources after the test


======================================== 4 passed in 0.02s ========================================


<ExitCode.OK: 0>

In [20]:
import pytest
import ipytest
import sys

In [22]:
%%ipytest -v -s
import logging
logging.basicConfig(level=logging.INFO)

def test_one():
    print("Executing test_one")
    assert True

def test_two():
    print("Executing test_two")
    assert True

======================================= test session starts =======================================
platform win32 -- Python 3.13.9, pytest-8.4.2, pluggy-1.5.0
rootdir: d:\Learning\Projects\TimePass\DSA
plugins: anyio-4.10.0, cov-7.1.0
collected 2 items

t_f58b8dd5a3c941419e527b68f1402cb9.py Executing test_one
.Executing test_two
.

======================================== 2 passed in 0.01s ========================================


In [35]:
import pytest
import ipytest
import sys



ipytest.autoconfig()




@pytest.fixture(scope="class", autouse=True)
def setup_cleanup():
    print("Setting up resources before the test")
    yield
    print("Cleaning up resources after the test")

class TestExampleAutouse:
    def test_one(self):
        print("Executing test_one")
        assert True

    def test_two(self ):
        print("Executing test_two")
        assert True




ipytest.run("-v","-s" )



======================================= test session starts =======================================
platform win32 -- Python 3.13.9, pytest-8.4.2, pluggy-1.5.0
rootdir: d:\Learning\Projects\TimePass\DSA
plugins: anyio-4.10.0, cov-7.1.0
collected 6 items

t_f58b8dd5a3c941419e527b68f1402cb9.py Setting up resources before the test
Executing test_one
.Cleaning up resources after the test
Setting up resources before the test
Executing test_two
.Cleaning up resources after the test
Setting up resources before the test
Executing test_one
.Executing test_two
.Cleaning up resources after the test
Setting up resources before the test
Executing test_one
.Executing test_two
.Cleaning up resources after the test


======================================== 6 passed in 0.02s ========================================


<ExitCode.OK: 0>

In [38]:
# Problem 7: Class based decorator.
# Concept: Implementing a decorator as a class using __call__
import time
from functools import wraps
#Rewrite the @retry decorator from Problem 2 as a class instead of a function. The class should accept times as a constructor argument.
class Retry:
    def __init__(self,times=3):
        self.times=times


    def __call__(self,func):
        @wraps(func)
        def func_wrapper(*args,**kwargs):
            for i in range(self.times):
                try:
                    return func(*args,**kwargs)
                except Exception as e:
                    if i == self.times - 1:
                        raise e
        return func_wrapper

@Retry(times=3)
def flaky_function():
    if time.time() % 2 < 1:  # Simulate a failure half the time
        raise ValueError("Function failed")
    return "Function succeeded"


print(flaky_function())  # This will retry up to 3 times if it fails



Function succeeded


In [39]:
from queue import PriorityQueue

class Node:
    def __init__(self, value, priority):
        self.value = value
        self.priority = priority

    def __lt__(self, other):
        return self.priority < other.priority   
    
class PriorityQueueWithUpdate(PriorityQueue):
    def update(self, value, new_priority):
        # Remove the old node and add a new one with updated priority
        temp_list = []
        while not self.empty():
            node = self.get()
            if node.value == value:
                node.priority = new_priority
            temp_list.append(node)
        for node in temp_list:
            self.put(node)

    def get_all(self):
        items = []
        while not self.empty():
            items.append(self.get())
        for item in items:
            self.put(item)
        return items        
